In [1]:
!pip install -q feedparser beautifulsoup4 tqdm sentence-transformers umap-learn hdbscan xlsxwriter bitsandbytes accelerate transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.2 MB/s eta 0:00:00:00:0100:01


In [2]:
import requests
import feedparser
import pandas as pd
import numpy as np
import json
from collections import defaultdict
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from tqdm import tqdm
import re

from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import umap
import hdbscan

from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

session = requests.Session()

2026-06-17 08:27:09.656625: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781684830.061003      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781684830.173087      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781684831.157859      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781684831.157900      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781684831.157903      57 computation_placer.cc:177] computation placer alr

In [3]:
TOPIC_RSS = {

    "world": [
        "https://vnexpress.net/rss/the-gioi.rss",
        "https://thanhnien.vn/rss/the-gioi.rss",
        "https://tuoitre.vn/rss/the-gioi.rss",
        "https://dantri.com.vn/rss/the-gioi.rss",
        "https://infonet.vietnamnet.vn/rss/the-gioi.rss"
    ],

    "sports": [
        "https://vnexpress.net/rss/the-thao.rss",
        "https://thanhnien.vn/rss/the-thao.rss",
        "https://tuoitre.vn/rss/the-thao.rss",
        "https://dantri.com.vn/rss/the-thao.rss",
        "https://infonet.vietnamnet.vn/rss/the-thao.rss"
    ],

    "technology": [
        "https://vnexpress.net/rss/so-hoa.rss",
        "https://thanhnien.vn/rss/cong-nghe.rss",
        "https://tuoitre.vn/rss/nhip-song-so.rss",
        "https://dantri.com.vn/rss/cong-nghe.rss",
        "https://infonet.vietnamnet.vn/rss/cong-nghe.rss"
    ],

    "business": [
        "https://vnexpress.net/rss/kinh-doanh.rss",
        "https://thanhnien.vn/rss/tai-chinh-kinh-doanh.rss",
        "https://tuoitre.vn/rss/kinh-doanh.rss",
        "https://dantri.com.vn/rss/kinh-doanh.rss",
        "https://infonet.vietnamnet.vn/rss/kinh-doanh.rss"
    ],

    "entertainment": [
        "https://vnexpress.net/rss/giai-tri.rss",
        "https://thanhnien.vn/rss/giai-tri.rss",
        "https://tuoitre.vn/rss/giai-tri.rss",
        "https://dantri.com.vn/rss/giai-tri.rss",
        "https://infonet.vietnamnet.vn/rss/giai-tri.rss"
    ],

    "education": [
        "https://vnexpress.net/rss/giao-duc.rss",
        "https://thanhnien.vn/rss/giao-duc.rss",
        "https://tuoitre.vn/rss/giao-duc.rss",
        "https://dantri.com.vn/rss/giao-duc.rss",
        "https://infonet.vietnamnet.vn/rss/giao-duc.rss"
    ]
}

In [4]:
def collect_articles(topic_rss, limit_per_feed=20):

    articles = []

    for topic, rss_list in topic_rss.items():

        for rss_url in rss_list:

            try:

                feed = feedparser.parse(rss_url)

                for entry in feed.entries[:limit_per_feed]:

                    articles.append({

                        "topic": topic,
                        "rss_source": rss_url,
                        "title": entry.get("title", ""),
                        "url": entry.get("link", ""),
                        "published": entry.get("published", "")

                    })

            except Exception as e:

                print("RSS error:", rss_url)

    return pd.DataFrame(articles)

In [5]:
rss_df = collect_articles(
    TOPIC_RSS,
    limit_per_feed=50
)

rss_df.info()
rss_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1411 entries, 0 to 1410
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   topic       1411 non-null   object
 1   rss_source  1411 non-null   object
 2   title       1411 non-null   object
 3   url         1411 non-null   object
 4   published   1411 non-null   object
dtypes: object(5)
memory usage: 55.2+ KB


,topic,rss_source,title,url,published
0,world,https://vnexpress.net/rss/the-gioi.rss,CĐV Scotland 'càn quét' quán bar ở Boston,https://vnexpress.net/cdv-scotland-can-quet-qu...,"Wed, 17 Jun 2026 14:51:34 +0700"
1,world,https://vnexpress.net/rss/the-gioi.rss,"Cảnh sát Mỹ bắn vào xe, bé một tuổi thiệt mạng",https://vnexpress.net/canh-sat-my-ban-vao-xe-b...,"Wed, 17 Jun 2026 14:00:44 +0700"
2,world,https://vnexpress.net/rss/the-gioi.rss,"Cường kích Su-24 Ukraine rơi, hai phi công thi...",https://vnexpress.net/cuong-kich-su-24-ukraine...,"Wed, 17 Jun 2026 13:47:55 +0700"
3,world,https://vnexpress.net/rss/the-gioi.rss,Tàu dầu Iran rời khu vực bị Mỹ phong tỏa,https://vnexpress.net/tau-dau-iran-roi-khu-vuc...,"Wed, 17 Jun 2026 12:26:21 +0700"
4,world,https://vnexpress.net/rss/the-gioi.rss,Cú trượt dài vào vòng lao lý của con riêng Thá...,https://vnexpress.net/cu-truot-dai-vao-vong-la...,"Wed, 17 Jun 2026 12:02:01 +0700"


In [6]:
def parse_vnexpress(soup):

    title = soup.select_one("h1.title-detail")
    description = soup.select_one("p.description")
    paragraphs = soup.select("article.fck_detail p.Normal")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [7]:
def parse_thanhnien(soup):

    title = soup.select_one("h1.detail-title")
    description = soup.select_one("h2.detail-sapo")
    paragraphs = soup.select(".detail-content > :is(p, h1, h2, h3, h4, h5, h6)")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [8]:
def parse_tuoitre(soup):

    title = soup.select_one("h1.article-title")
    description = soup.select_one("h2.detail-sapo")
    paragraphs = soup.select(".detail-content > :is(p, h1, h2, h3, h4, h5, h6)")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [9]:
def parse_dantri(soup):

    title = soup.select_one("h1.e-magazine__title")
    description = soup.select_one("h2.e-magazine__sapo")
    paragraphs = soup.select(".e-magazine__body > :is(p, h1, h2, h3, h4, h5, h6)")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [10]:
def parse_vietnamnet(soup):

    title = soup.select_one("h1.contentDetail-title")
    description = soup.select_one(".contentDetail-sapo")
    paragraphs = soup.select(".contentDetail__main-reading > :is(p, h1, h2, h3, h4, h5, h6)")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [11]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

def get_article_content(url):
    res = session.get(
        url,
        headers=headers,
        timeout=20
    )

    res.encoding = "utf-8"

    soup = BeautifulSoup(res.text, "html.parser")

    if "vnexpress.net" in url:
        parsed = parse_vnexpress(soup)

    elif "thanhnien.vn" in url:
        parsed = parse_thanhnien(soup)

    elif "tuoitre.vn" in url:
        parsed = parse_tuoitre(soup)

    elif "dantri.com.vn" in url:
        parsed = parse_dantri(soup)

    elif "infonet.vietnamnet.vn" in url:
        parsed = parse_vietnamnet(soup)

    else:
        return None

    parsed["url"] = url

    parsed["full_text"] = "\n".join([
        parsed["title"],
        parsed["description"],
        parsed["content"]
    ])

    parsed["summary"] = "\n".join([
        parsed["title"],
        parsed["description"],
        parsed["content"][:1000]
    ])

    return parsed

In [12]:
def extract_source_name(rss_url):

    if "vnexpress" in rss_url:
        return "VnExpress"

    if "thanhnien" in rss_url:
        return "Thanh Niên"

    if "tuoitre" in rss_url:
        return "Tuổi Trẻ"

    if "dantri" in rss_url:
        return "Dân Trí"

    if "vietnamnet" in rss_url:
        return "Vietnamnet"

    return rss_url

In [13]:
def fetch_article(row):

    try:

        article = get_article_content(row["url"])

        if article and len(article["full_text"]) > 300:

            article["topic"] = row["topic"]
            article["rss_source"] = row["rss_source"]

            # ADD
            article["published"] = row.get("published", "")
            article["source"] = extract_source_name(row["rss_source"])

            return article

    except Exception:

        return None

In [14]:
data = []

rows = list(rss_df.to_dict("records"))

with ThreadPoolExecutor(max_workers=64) as executor:

    futures = [
        executor.submit(fetch_article, row)
        for row in rows
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        result = future.result()

        if result is not None:
            data.append(result)

df = pd.DataFrame(data)

print(df.shape)

100%|██████████| 1411/1411 [02:44<00:00,  8.57it/s]

(1084, 10)


In [15]:
embed_model = SentenceTransformer(
    "BAAI/bge-m3"
)

embeddings = embed_model.encode(
    df["summary"].tolist(),
    show_progress_bar=True,
    batch_size=128
)

print(embeddings.shape)

embeddings = normalize(embeddings, norm="l2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

(1084, 1024)


## Low-noise clustering patch

Notebook này đã được chỉnh để giảm `noise_ratio` bằng cách chạy HDBSCAN trên không gian UMAP 30 chiều và tự động quét nhiều cấu hình `min_cluster_size`, `min_samples`, `cluster_selection_method`. Cấu hình tốt nhất được chọn theo objective ưu tiên độ phủ cụm cao, silhouette tốt, đồng thời tránh cụm khổng lồ hoặc quá nhiều cụm vụn.

Sau khi chạy lại từ cell embedding trở xuống, file `trends.json` sẽ được export lại với các metric mới: `noise_ratio`, `cluster_coverage`, `clustered_articles`, `noise_articles`, `silhouette_score`, `dbcv`, `davies_bouldin_index`.


In [16]:
# =========================================================
# OPTIMIZED CLUSTERING: UMAP + HDBSCAN AUTO-SWEEP
# =========================================================
# Mục tiêu:
# - Giảm noise_ratio so với cấu hình cũ.
# - Không ép toàn bộ bài báo vào 1 cụm khổng lồ.
# - Ưu tiên cụm có độ phủ tốt nhưng vẫn giữ tính tách biệt chủ đề.
#
# Lưu ý:
# - embeddings vẫn là vector BAAI/bge-m3 đã normalize L2.
# - UMAP chỉ dùng để tạo không gian clustering ổn định hơn.
# - Khi chọn bài đại diện cho LLM, vẫn dùng embeddings gốc để giữ ngữ nghĩa.

RANDOM_STATE = 42

reducer = umap.UMAP(
    n_neighbors=15,
    n_components=30,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE
)

embedding_2d = reducer.fit_transform(embeddings)
df["umap_x"] = embedding_2d[:, 0]
df["umap_y"] = embedding_2d[:, 1]

USE_UMAP_FOR_CLUSTERING = True

if USE_UMAP_FOR_CLUSTERING:
    cluster_input = embedding_2d
    print("UMAP clustering input:", cluster_input.shape)

else:
    cluster_input = embeddings
    print("Raw embedding clustering input:", cluster_input.shape)

def summarize_labels(candidate_labels):
    unique_clusters = [x for x in np.unique(candidate_labels) if x != -1]

    cluster_sizes = [
        int(np.sum(candidate_labels == cid))
        for cid in unique_clusters
    ]

    n_clusters = len(unique_clusters)
    noise_articles = int(np.sum(candidate_labels == -1))
    noise_ratio = float(np.mean(candidate_labels == -1))
    clustered_articles = int(np.sum(candidate_labels != -1))
    largest_cluster_ratio = (
        max(cluster_sizes) / len(candidate_labels)
        if cluster_sizes
        else 0
    )

    return {
        "n_clusters": n_clusters,
        "clustered_articles": clustered_articles,
        "noise_articles": noise_articles,
        "noise_ratio": noise_ratio,
        "largest_cluster_ratio": largest_cluster_ratio,
        "cluster_sizes": cluster_sizes
    }


def safe_silhouette(X, candidate_labels):
    mask = candidate_labels != -1
    valid_labels = candidate_labels[mask]

    if len(set(valid_labels)) < 2 or len(valid_labels) < 3:
        return None

    try:
        return float(
            silhouette_score(
                X[mask],
                valid_labels,
                metric="euclidean"
            )
        )
    except Exception:
        return None


candidate_configs = [
    # Cấu hình giảm noise mạnh, phù hợp dashboard/trend discovery
    {"min_cluster_size": 3, "min_samples": 1, "cluster_selection_method": "eom", "cluster_selection_epsilon": 0.00},
    {"min_cluster_size": 3, "min_samples": 1, "cluster_selection_method": "eom", "cluster_selection_epsilon": 0.05},
    {"min_cluster_size": 3, "min_samples": 1, "cluster_selection_method": "leaf", "cluster_selection_epsilon": 0.00},

    # Cấu hình cân bằng hơn
    {"min_cluster_size": 4, "min_samples": 1, "cluster_selection_method": "eom", "cluster_selection_epsilon": 0.00},
    {"min_cluster_size": 4, "min_samples": 1, "cluster_selection_method": "eom", "cluster_selection_epsilon": 0.05},
    {"min_cluster_size": 4, "min_samples": 2, "cluster_selection_method": "eom", "cluster_selection_epsilon": 0.00},

    # Cấu hình bảo thủ hơn để tránh cụm rác
    {"min_cluster_size": 5, "min_samples": 1, "cluster_selection_method": "eom", "cluster_selection_epsilon": 0.00},
    {"min_cluster_size": 5, "min_samples": 2, "cluster_selection_method": "eom", "cluster_selection_epsilon": 0.00},
    {"min_cluster_size": 6, "min_samples": 1, "cluster_selection_method": "eom", "cluster_selection_epsilon": 0.00},
]

candidate_results = []

for cfg in candidate_configs:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=cfg["min_cluster_size"],
        min_samples=cfg["min_samples"],
        metric="euclidean",
        cluster_selection_method=cfg["cluster_selection_method"],
        cluster_selection_epsilon=cfg["cluster_selection_epsilon"]
    )

    candidate_labels = clusterer.fit_predict(cluster_input)
    summary = summarize_labels(candidate_labels)
    sil = safe_silhouette(cluster_input, candidate_labels)

    # Score chọn cấu hình:
    # - ưu tiên coverage cao / noise thấp
    # - cộng điểm nếu silhouette tốt
    # - phạt nếu tạo cụm khổng lồ hoặc quá ít/quá nhiều cụm
    coverage = 1 - summary["noise_ratio"]
    sil_score = max(sil, 0) if sil is not None else 0

    objective = 0.70 * coverage + 0.20 * sil_score

    if summary["n_clusters"] < 8:
        objective -= 0.20

    if summary["n_clusters"] > 80:
        objective -= 0.004 * (summary["n_clusters"] - 80)

    if summary["largest_cluster_ratio"] > 0.35:
        objective -= 0.60 * (summary["largest_cluster_ratio"] - 0.35)

    result = {
        **cfg,
        **summary,
        "silhouette": sil,
        "objective": objective,
        "labels": candidate_labels,
        "clusterer": clusterer
    }

    candidate_results.append(result)

candidate_results = sorted(
    candidate_results,
    key=lambda x: x["objective"],
    reverse=True
)

print("=== HDBSCAN CONFIG SEARCH RESULTS ===")
for i, r in enumerate(candidate_results, 1):
    print(
        i,
        "| size=", r["min_cluster_size"],
        "| samples=", r["min_samples"],
        "| method=", r["cluster_selection_method"],
        "| eps=", f'{r["cluster_selection_epsilon"]:.2}',
        "| clusters=", f'{r["n_clusters"]:3}',
        "| noise=", f'{r["noise_ratio"]:.2%}',
        "| largest=", f'{r["largest_cluster_ratio"]:.2%}',
        "| silhouette=", None if r["silhouette"] is None else f'{r["silhouette"]:.4}',
        "| objective=", f'{r["objective"]:.4}',
    )

best_result = candidate_results[0]
df["cluster"] = labels = best_result["labels"]

print("\n=== SELECTED CONFIG ===")
print({
    "min_cluster_size": best_result["min_cluster_size"],
    "min_samples": best_result["min_samples"],
    "cluster_selection_method": best_result["cluster_selection_method"],
    "cluster_selection_epsilon": f'{best_result["cluster_selection_epsilon"]:.2}',
    "num_clusters": best_result["n_clusters"],
    "noise_ratio": f'{best_result["noise_ratio"]:.2%}',
    "clustered_articles": best_result["clustered_articles"],
    "noise_articles": best_result["noise_articles"],
    "largest_cluster_ratio": f'{best_result["largest_cluster_ratio"]:.2%}',
    "silhouette": f'{best_result["silhouette"]:.4}',
})


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP clustering input: (1084, 30)
=== HDBSCAN CONFIG SEARCH RESULTS ===
1 | size= 6 | samples= 1 | method= eom | eps= 0.0 | clusters=  65 | noise= 13.19% | largest= 4.34% | silhouette= 0.5191 | objective= 0.7115
2 | size= 5 | samples= 2 | method= eom | eps= 0.0 | clusters=  71 | noise= 14.30% | largest= 5.26% | silhouette= 0.5199 | objective= 0.7039
3 | size= 5 | samples= 1 | method= eom | eps= 0.0 | clusters=  82 | noise= 14.21% | largest= 4.34% | silhouette= 0.5066 | objective= 0.6939
4 | size= 4 | samples= 2 | method= eom | eps= 0.0 | clusters=  91 | noise= 16.79% | largest= 4.34% | silhouette= 0.5222 | objective= 0.6429
5 | size= 4 | samples= 1 | method= eom | eps= 0.0 | clusters= 100 | noise= 13.93% | largest= 4.34% | silhouette= 0.5055 | objective= 0.6236
6 | size= 4 | samples= 1 | method= eom | eps= 0.05 | clusters= 100 | noise= 13.93% | largest= 4.34% | silhouette= 0.5055 | objective= 0.6236
7 | size= 3 | samples= 1 | method= eom | eps= 0.0 | clusters= 152 | noise= 12.92% | lar

In [17]:
# =========================================================
# CLUSTERING EVALUATION
# =========================================================

mask = labels != -1
valid_labels = labels[mask]

num_clusters = int(len([x for x in np.unique(labels) if x != -1]))
noise_articles = int(np.sum(labels == -1))
clustered_articles = int(np.sum(labels != -1))
noise_ratio = float(np.mean(labels == -1))
cluster_coverage = float(1 - noise_ratio)

print(f"Number of clusters: {num_clusters}")
print(f"Clustered articles: {clustered_articles}")
print(f"Noise articles: {noise_articles}")
print(f"Noise ratio: {noise_ratio:.2%}")
print(f"Cluster coverage: {cluster_coverage:.2%}")

# Dùng cluster_input để đánh giá vì HDBSCAN chạy trên không gian này.
if len(set(valid_labels)) >= 2 and len(valid_labels) >= 3:
    silhouette = silhouette_score(
        cluster_input[mask],
        valid_labels,
        metric="euclidean"
    )
else:
    silhouette = None

print(f"Silhouette score: {silhouette:.4}")

try:
    dbcv_score = hdbscan.validity_index(
        cluster_input[mask].astype(np.float64),
        valid_labels,
        metric="euclidean"
    )
except Exception:
    dbcv_score = None

print(f"DBCV: {dbcv_score:.4}")

try:
    db_index = davies_bouldin_score(
        cluster_input[mask],
        valid_labels
    )
except Exception:
    db_index = None

print(f"Davies-Bouldin Index: {db_index:.4}")


def cluster_coherence(cluster_embs):
    centroid = cluster_embs.mean(axis=0)

    sims = cosine_similarity(
        cluster_embs,
        [centroid]
    )

    return sims.mean()


cluster_scores = []

for cid in np.unique(labels):

    if cid == -1:
        continue

    # Coherence dùng embeddings gốc để đo tính nhất quán ngữ nghĩa
    cluster_embs = embeddings[labels == cid]

    if len(cluster_embs) < 2:
        continue

    score = cluster_coherence(cluster_embs)

    cluster_scores.append({
        "cluster": int(cid),
        "size": int(len(cluster_embs)),
        "coherence": float(score)
    })

cluster_scores = sorted(
    cluster_scores,
    key=lambda x: x["coherence"],
    reverse=True
)

print("\nTop coherence clusters:")
for row in cluster_scores[:10]:
    print(row)

if cluster_scores:
    weighted_mean = np.average(
        [x["coherence"] for x in cluster_scores],
        weights=[x["size"] for x in cluster_scores]
    )
else:
    weighted_mean = None

print(f"Weighted mean coherence: {weighted_mean:.4}")


Number of clusters: 65
Clustered articles: 941
Noise articles: 143
Noise ratio: 13.19%
Cluster coverage: 86.81%
Silhouette score: 0.5191
DBCV: 0.42565127107807527
Davies-Bouldin Index: 0.6817

Top coherence clusters:
{'cluster': 11, 'size': 8, 'coherence': 0.9080218076705933}
{'cluster': 23, 'size': 12, 'coherence': 0.8633783459663391}
{'cluster': 14, 'size': 9, 'coherence': 0.854378342628479}
{'cluster': 20, 'size': 8, 'coherence': 0.8462567925453186}
{'cluster': 17, 'size': 8, 'coherence': 0.830312967300415}
{'cluster': 58, 'size': 8, 'coherence': 0.8260408043861389}
{'cluster': 2, 'size': 18, 'coherence': 0.8238078951835632}
{'cluster': 19, 'size': 10, 'coherence': 0.8192057609558105}
{'cluster': 16, 'size': 21, 'coherence': 0.8180857300758362}
{'cluster': 27, 'size': 14, 'coherence': 0.815883219242096}
Weighted mean coherence: 0.7514


In [18]:
for cluster_id in sorted(df["cluster"].unique())[:20]:

    print("=" * 80)
    print("CLUSTER", cluster_id)

    subset = df[df["cluster"] == cluster_id]

    for title in subset["title"].head(5):
        print("-", title)

CLUSTER -1
- CĐV Scotland 'càn quét' quán bar ở Boston
- Cảnh sát Mỹ bắn vào xe, bé một tuổi thiệt mạng
- Lãnh đạo và nhân viên Starbucks Hàn Quốc phải học lại lịch sử
- Trung Quốc - 'nam châm' hút các gia đình giàu có vùng Vịnh
- Cường kích Su-24 Ukraine rơi, hai phi công thiệt mạng
CLUSTER 0
- Vòng 24 Ngoại hạng Anh: HLV Frank Lampard hài lòng với 1 điểm trước Arsenal
- Vòng 25 Premier League: M.U bỏ lỡ cơ hội vàng để đuổi Chelsea
- Thua Man City với tỷ số không tưởng, đội bóng của John Terry lâm nguy
- Liverpool lập những kỷ lục khó tin, HLV Jose Mourinho đổ lỗi cho trọng tài VAR
- Lịch thi đấu vòng 22 Premier League: Tottenham vs Liverpool sẽ là trận tâm điểm
CLUSTER 1
- FPT Play chiếu trọn vẹn 3 giải bóng đá nóng nhất hè 2026
- Đội tuyển Việt Nam thể hình đẹp như siêu mẫu lại cực khỏe, mục tiêu đâu chỉ mỗi AFF Cup
- Lịch thi đấu đội tuyển Việt Nam mới nhất: So tài Myanmar, chưa lộ diện kênh phát trực tiếp
- Báo Thái Lan phân tích cực sâu về đội tuyển Việt Nam, bất an một chuyện…
-

In [19]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [20]:
def generate_batch(prompts, batch_size=4):

    results = []

    system_prompt = (
        "Bạn là AI chuyên tổng hợp xu hướng báo chí.\n"
        "Chỉ trả về đúng 1 câu summary.\n"
        "Không giải thích.\n"
        "Không bullet point.\n"
        "Không nhắc lại yêu cầu."
    )

    for i in tqdm(range(0, len(prompts), batch_size)):

        batch_prompts = prompts[i:i + batch_size]

        # ==========================================
        # BUILD CHAT MESSAGES
        # ==========================================

        message_batch = []

        for prompt in batch_prompts:

            messages = [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]

            message_batch.append(messages)

        # ==========================================
        # APPLY CHAT TEMPLATE
        # ==========================================

        text_batch = tokenizer.apply_chat_template(
            message_batch,
            tokenize=False,
            add_generation_prompt=True
        )

        # ==========================================
        # TOKENIZE
        # ==========================================

        inputs = tokenizer(
            text_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)

        # ==========================================
        # GENERATE
        # ==========================================

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

        # ==========================================
        # REMOVE PROMPT TOKENS
        # ==========================================

        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids
            in zip(inputs.input_ids, outputs)
        ]

        # ==========================================
        # DECODE
        # ==========================================

        decoded = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        results.extend([
            d.strip()
            for d in decoded
        ])

        torch.cuda.empty_cache()

    return results

In [22]:
# =========================================================
# CONFIG
# =========================================================

TOP_K = 5
DIVERSITY_THRESHOLD = 0.9

# =========================================================
# HELPER
# =========================================================

def normalize(v):
    norm = np.linalg.norm(v)

    if norm == 0:
        return v

    return v / norm


def select_representative_docs(
    cluster_embeddings,
    cluster_indices,
    top_k=5,
    diversity_threshold=0.9
):
    """
    Chọn top-k bài:
    - gần centroid
    - có diversity nhẹ
    """

    centroid = np.mean(cluster_embeddings, axis=0)
    centroid = normalize(centroid)

    normalized_embs = np.array([
        normalize(x)
        for x in cluster_embeddings
    ])

    sims = cosine_similarity(
        [centroid],
        normalized_embs
    )[0]

    sorted_idx = np.argsort(-sims)

    selected_local_idx = []

    for idx in sorted_idx:

        current_emb = normalized_embs[idx]

        too_similar = False

        for selected_idx in selected_local_idx:

            sim = cosine_similarity(
                [current_emb],
                [normalized_embs[selected_idx]]
            )[0][0]

            if sim >= diversity_threshold:
                too_similar = True
                break

        if not too_similar:
            selected_local_idx.append(idx)

        if len(selected_local_idx) >= top_k:
            break

    selected_global_indices = [
        cluster_indices[i]
        for i in selected_local_idx
    ]

    return selected_global_indices


def build_trend_prompt(representative_articles):

    prompt = """Tóm tắt các bài báo sau thành đúng 1 câu tiếng Việt ngắn.

Không bullet point.
Không giải thích.
Không nhắc lại yêu cầu.

Các bài báo:
"""

    for i, article in enumerate(representative_articles, 1):

        prompt += f"""
{i}.
Tiêu đề: {article["title"]}

Tóm tắt:
{article["description"]}
"""

    prompt += "\nKết quả:\n"
    return prompt


# =========================================================
# BUILD CLUSTER DATA
# =========================================================

cluster_data = {}

cluster_docs = defaultdict(list)

for idx, row in df.iterrows():
    cluster_docs[row["cluster"]].append(idx)

for cluster_id, indices in cluster_docs.items():
    # skip noise
    if cluster_id == -1:
        continue

    subset = df.iloc[indices]

    cluster_embs = embeddings[indices]

    # =====================================================
    # SELECT REPRESENTATIVE ARTICLES
    # =====================================================

    representative_indices = select_representative_docs(
        cluster_embs,
        indices,
        top_k=TOP_K,
        diversity_threshold=DIVERSITY_THRESHOLD
    )

    all_articles = []
    representative_articles = []
    
    for idx in indices:
        row = df.iloc[idx]

        data_to_save = {
            "index": int(idx),
            "title": row["title"],
            "description": row["description"],
            "url": row["url"],
            "topic": row["topic"],
            "published": row.get("published", ""),
            "source": row.get("source", row.get("rss_source", "")),
            "umap_x": float(row["umap_x"]),
            "umap_y": float(row["umap_y"]),
        }

        all_articles.append(data_to_save)
        if idx in representative_indices:
            representative_articles.append(data_to_save)

    # =====================================================
    # BUILD PROMPT
    # =====================================================

    prompt = build_trend_prompt(
        representative_articles
    )

    # =====================================================
    # SAVE CLUSTER
    # =====================================================

    cluster_data[cluster_id] = {
        "cluster_id": int(cluster_id),
        "num_articles": len(indices),
        "article_indices": indices,
        "representative_indices": representative_indices,
        "representative_articles": representative_articles,
        "all_articles": all_articles,
        "prompt": prompt,
        "trend": None
    }

# =========================================================
# GENERATE TRENDS
# =========================================================

cluster_ids = list(cluster_data.keys())

prompts = [
    cluster_data[cid]["prompt"]
    for cid in cluster_ids
]

trends = generate_batch(
    prompts,
    batch_size=8
)

for cid, trend in zip(cluster_ids, trends):
    cluster_data[cid]["trend"] = trend

# =========================================================
# SORT
# =========================================================

sorted_clusters = sorted(
    cluster_data.items(),
    key=lambda x: x[1]["num_articles"],
    reverse=True
)

cluster_data = {
    i: data
    for i, (_, data) in enumerate(sorted_clusters)
}

# =========================================================
# PREVIEW
# =========================================================

for cid, data in list(cluster_data.items())[:20]:
    print("=" * 80)
    print("CLUSTER:", cid)

    print("TREND:")
    print(data["trend"])

    print(f"\nNUMBER OF ARTICLES: {data['num_articles']}")

    print("\nREPRESENTATIVE ARTICLES:")
    for article in data["representative_articles"]:

        print("-", article["title"])


100%|██████████| 9/9 [02:09<00:00, 14.41s/it]

CLUSTER: 0
TREND:
U23 Việt Nam cần chiến thắng trong trận cuối và hy vọng vào kết quả từ trận đấu giữa U23 UAE và U23 Jordan để lọt vào vòng trong U23 châu Á 2020.

NUMBER OF ARTICLES: 47

REPRESENTATIVE ARTICLES:
- Đình Trọng: "Chúng tôi có cơ hội để bước vào vòng tiếp theo"
- Hòa 0-0 trước U23 Jordan, thầy trò HLV Park Hang Seo mất quyền tự quyết
- 20h15' ngày 13/1, U23 Việt Nam - U23 Jordan: Sẽ có 3 điểm trước đội bóng Tây Á?
- HLV Park Hang Seo: "Ở trận đấu cuối cùng, U23 Việt Nam cần một chiến thắng"
- Ông Park tin vào kinh nghiệm đối đầu các đội bóng Tây Á
CLUSTER: 1
TREND:
Messi ghi ba bàn giúp Argentina thắng Algeria 3-0 trong trận ra quân World Cup 2026.

NUMBER OF ARTICLES: 45

REPRESENTATIVE ARTICLES:
- Messi lập hattrick, Argentina đại thắng trận ra quân World Cup
- World Cup 2026, Argentina 3-0 Algeria: Cú hat-trick lịch sử của Messi
- Hãy tận hưởng giây phút Messi còn cùng Argentina sải bước ở World Cup
- Messi lập siêu phẩm, xô đổ loạt cột mốc lịch sử trong ngày ra quân 

In [23]:
export_trends = []

for rank, data in cluster_data.items():
    export_trends.append({
        "rank": int(rank) + 1,
        "cluster_id": int(data["cluster_id"]),
        "trend_name": data["trend"],
        "article_count": int(data["num_articles"]),
        "trend": data["trend"],
        "summary": data["trend"],
        "representative_articles": data["representative_articles"],
        "articles": data["all_articles"]
    })

payload = {
    "project_name": "News Trend Detection",
    "generated_at": datetime.now().isoformat(),
    "model": {
        "embedding_model": "BAAI/bge-m3",
        "llm_model": "Qwen/Qwen2.5-7B-Instruct",
        "quantization": "4-bit"
    },
    "clustering_config": {
        "use_umap_for_clustering": bool(USE_UMAP_FOR_CLUSTERING),
        "selected_hdbscan": {
            "min_cluster_size": int(best_result["min_cluster_size"]),
            "min_samples": int(best_result["min_samples"]),
            "cluster_selection_method": best_result["cluster_selection_method"],
            "cluster_selection_epsilon": float(best_result["cluster_selection_epsilon"])
        }
    },
    "metrics": {
        "total_rss_articles": int(len(rss_df)),
        "total_scraped_articles": int(len(df)),
        "num_clusters": int(num_clusters),
        "clustered_articles": int(clustered_articles),
        "noise_articles": int(noise_articles),
        "noise_ratio": float(noise_ratio),
        "cluster_coverage": float(cluster_coverage),
        "weighted_coherence": None if weighted_mean is None else float(weighted_mean),
        "silhouette_score": None if silhouette is None else float(silhouette),
        "dbcv": None if dbcv_score is None else float(dbcv_score),
        "davies_bouldin_index": None if db_index is None else float(db_index)
    },
    "trends": export_trends
}

with open("trends.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Exported trends.json")
print("Metrics:")
print(json.dumps(payload["metrics"], ensure_ascii=False, indent=2))


Exported trends.json
Metrics:
{
  "total_rss_articles": 1411,
  "total_scraped_articles": 1084,
  "num_clusters": 65,
  "clustered_articles": 941,
  "noise_articles": 143,
  "noise_ratio": 0.13191881918819187,
  "cluster_coverage": 0.8680811808118081,
  "weighted_coherence": 0.7514157845033976,
  "silhouette_score": 0.519071638584137,
  "dbcv": 0.42565127107807527,
  "davies_bouldin_index": 0.6817176657929579
}
